# K-Nearest Neighbors for weather risk 

Multidimesnional K-nn classifier to determine if certain weather is safe/unsafe for planes to fly through

## Do these in order 
1. Balance dataset 
2. Training/testing split 
3. Normalizing 
4. Wrap KNN in KNeighborsClassifier(n_neighbors=5) ? maybe tune value of n later?  
5. Fit the model to the x_train and y_train ( these come from the normalized data)
6. Get/visualzie predictions(Can use MatPlotLib for this) - also print out accuracy of model to see if we cna fine tune 
7. (if we have time) - use predict_proba to convert the binary safe/unsafe tags into actual weather risk. 
8. Pass back this weather risk into calcuations.py file 

## 0. Imports

Uncomment or add any extra imports you need.

In [8]:
%pip install numpy pandas scikit-learn
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

Note: you may need to restart the kernel to use updated packages.


## 1. Configuration and loading data

**TODO:** Set `CSV_PATH` to balanced or raw dataset. Run `ml_dataset_scripts.py` from this folder first if you need to inspect counts; export a balanced CSV.

In [9]:
# Path relative to this notebook (usually backend/calculations)
CSV_PATH = Path("ml_ready_dataset.csv")
#print(Path("ml_ready_dataset (1).csv"))


#If the file is missing, create a tiny synthetic dataset 
if not CSV_PATH.is_file():
    print(f"Warning: {CSV_PATH} not found. Using synthetic data for pipeline practice.")
    rng = np.random.default_rng(42)
    n = 50000
    df = pd.DataFrame(
        {
            "wind_speed": rng.lognormal(3, 0.5, n),
            "wind_gust": rng.lognormal(3.2, 0.5, n),
            "precip_inches": rng.exponential(0.1, n),
            "humidity_pct": rng.uniform(20, 100, n),
            "lightning_strikes_10mi": rng.poisson(2, n),
            "unsafe_weather": rng.choice([0, 1], n, p=[0.85, 0.15]),
        }
    )
else:

    df = pd.read_csv(CSV_PATH)


print(df.shape)
print(df.dtypes)
df.head()

(50000, 6)
wind_speed                float64
wind_gust                 float64
precip_inches             float64
humidity_pct              float64
lightning_strikes_10mi      int64
unsafe_weather              int64
dtype: object


,wind_speed,wind_gust,precip_inches,humidity_pct,lightning_strikes_10mi,unsafe_weather
0,23.391169,36.047591,0.118434,24.352865,1,0
1,11.941359,9.757609,0.346706,55.852357,2,0
2,29.230877,21.762127,0.194788,86.372144,4,1
3,32.145818,14.220462,0.135452,60.905497,4,0
4,7.572191,102.701478,0.119545,60.829055,5,0


### Balance Dataset

### Split Data

In [10]:
# Split data here for training/testing - 80 -20 split 

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

data = df.drop(columns=['FL_DATE', 'ORIGIN', 'DEST', 'MKT_CARRIER'], errors='ignore')

# change to columns we want to use to train the data 
X = data.drop(columns=['unsafe_weather']) 

# change to the unsafe_weather/safe weather 
y = data['unsafe_weather']   


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) # tune later, but for now it is 80-20

print(X_test[0:5])
print(y)

       wind_speed  wind_gust  precip_inches  humidity_pct  \
33553   27.452868  41.283067       0.031494     47.966997   
9427    19.997861  46.576982       0.023258     23.493466   
199     18.509397  27.251731       0.141890     90.856886   
12447   21.735654  40.211352       0.274270     50.601958   
39489   23.234477  45.419415       0.085021     90.042235   

       lightning_strikes_10mi  
33553                       1  
9427                        2  
199                         1  
12447                       0  
39489                       2  
0        0
1        0
2        1
3        0
4        0
        ..
49995    0
49996    0
49997    0
49998    0
49999    0
Name: unsafe_weather, Length: 50000, dtype: int64


### Normalize Data


In [4]:
# Adding normalizing 

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

data_normalized  = (scaler.fit_transform(data)) # All column data is now normalized. 

print("max:" , scaler.data_max_)
print(data_normalized)


max: [245.57873679 233.5445629    1.24064937  99.99982904  12.
   1.        ]
[[0.08692936 0.14156401 0.09545706 0.05438946 0.08333333 0.        ]
 [0.03987684 0.02729256 0.279452   0.44814289 0.16666667 0.        ]
 [0.1109274  0.07947113 0.15700098 0.82964971 0.33333333 1.        ]
 ...
 [0.09154374 0.2109803  0.02326975 0.78708445 0.         0.        ]
 [0.15358878 0.08364659 0.01897594 0.70845491 0.33333333 0.        ]
 [0.15664529 0.13280222 0.00321706 0.2389184  0.08333333 0.        ]]


In [5]:
print(X_test.iloc[0])

wind_speed                27.452868
wind_gust                 41.283067
precip_inches              0.031494
humidity_pct              47.966997
lightning_strikes_10mi     1.000000
Name: 33553, dtype: float64


### KNN classifier 

In [ ]:
#Wrap KNN in KNeighborsClassifier(n_neighbors=5) ?
from sklearn.neighbors import KNeighborsClassifier
neigh = KNeighborsClassifier(n_neighbors=8) # May tune later 

neigh.fit(X_train, y_train) # Uncomment and use Pandas to extract all weather data(features) as X and Safe/unsafe weather column as Y 

for i in range(0, 99):
     print(neigh.predict([X_test.iloc[i]]), y_test.iloc[i])

#print(neigh.predict([X_test.iloc[0:99]])) # Fill in weather data to predict (for testing )

#print(y_test.iloc[i]) # Compare to actual value of safe/unsafe weather for that data point.

[0] 0
[0] 0
[0] 1
[0] 0
[0] 0
[0] 0
[0] 1
[0] 0
[0] 0
[0] 1
[0] 0
[0] 0
[1] 0
[0] 0
[0] 1
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0
[0] 1
[0] 0
[0] 0
[0] 0
[0] 1
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0
[0] 1
[0] 1
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0
[0] 1
[0] 1
[0] 0
[0] 1
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0
[0] 1
[0] 0
[0] 0
[0] 0
[0] 0
[0] 1
[0] 0
[0] 1
[0] 1
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0
[0] 1
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0
[0] 0


c:\Users\olive\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(
c:\Users\olive\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(
c:\Users\olive\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(
c:\Users\olive\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(
c:\Users\olive\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\uti

##### Visualization of Accuracy/Confusion Matrix to tune above


In [33]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# 1. Make predictions on test data
y_pred = neigh.predict(X_test)

# 2. Create a confusion matrix
cm = confusion_matrix(y_test, y_pred)
print(cm)

# 3. Display matrix visually
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=neigh.classes_)
print((cm[0,0] + cm[1,1]) / cm.sum())

[[8384   97]
 [1502   17]]
0.8401
